# V7 可视化测试 Notebook

本 notebook 测试 V7 系统的可视化能力：

| 可视化 | 类型 | 测试内容 |
|--------|------|----------|
| 市场状态3D散点图 | 3D plot | 估值/动量/体制三维分布 |
| 体制概率柱状图 | Bar | 各体制softmax概率 |
| 衍生品仪表盘 | 组合面板 | 期限结构 + 基差分析 |
| 风险仪表盘 | 组合面板 | 风险评分 + 雷达图 + 指标摘要 |
| PE百分位历史图 | 时间序列 | 估值百分位走势 |
| 多指数趋势图 | 多线图 | 基准指数MA趋势 |
| 综合仪表盘 | 大面板 | 多图合并一页 |


In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# 中文字体设置
font_path = '/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

import yaml
config_path = PROJECT_ROOT / 'config' / 'market_state_system' / 'system_config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 输出目录
VIZ_DIR = PROJECT_ROOT / 'output' / 'visualization'
VIZ_DIR.mkdir(parents=True, exist_ok=True)

print(f'可视化输出目录: {VIZ_DIR}')
print(f'Matplotlib 后端: {matplotlib.get_backend()}')

## 0. 准备测试数据

构建 V7 完整服务栈并运行管线获取结果数据

In [ ]:
# 构建服务栈 & 运行管线
from base_services.config_service import ConfigService
from base_services.cache_service import CacheService
from data_service.tdx_adapter import TDXAdapter
from data_service.database_reader import DatabaseReader
from data_service.ak_adapter import AKAdapter
from data_service.data_loader_service import DataLoadingService
from market_state_system.core.contract_manager import ContractManager
from market_state_system.core.market_regime_engine import MarketRegimeEngine
from market_state_system.core.market_state_classifier import MarketStateClassifier
from market_state_system.core.derivatives_signal_engine import DerivativesSignalEngine

cfg_svc = ConfigService(system_name='market_state_system', enable_hot_reload=False)
from config.global_settings import DATABASE_ENGINES, DB_POOL_CONFIG

tdx = TDXAdapter({'host': config.get('tdx', {}).get('hq_host', '180.153.18.170'),
                  'port': config.get('tdx', {}).get('hq_port', 7709),
                  'pool_size': 3, 'retry_count': 2})
db = DatabaseReader(DATABASE_ENGINES, DB_POOL_CONFIG)
ak = AKAdapter({'cache_enabled': True})
data_svc = DataLoadingService(cfg_svc, db, tdx, ak, CacheService(max_size=200, ttl=3600))

# 运行核心管线
cm = ContractManager(reference_date=datetime.now())
regime_engine = MarketRegimeEngine(data_svc, config)
classifier = MarketStateClassifier(data_svc, config, regime_engine)
deriv_engine = DerivativesSignalEngine(data_svc, cm, config)

# 获取结果
regime_result = regime_engine.detect_regime()
classification_result = classifier.classify()
deriv_report = deriv_engine.generate_report()

print(f'✅ 管线运行完成')
print(f'  体制: {regime_result["current_regime"]}')
print(f'  状态: {classification_result["market_state"]}')

## 1. StateVisualizer 基础可视化

使用 V7 内置的 StateVisualizer 生成标准图表

In [ ]:
from market_state_system.visualization.state_visualizer import StateVisualizer

viz = StateVisualizer({'output_dir': str(VIZ_DIR)})
print(f'StateVisualizer 初始化完成 | 输出目录: {viz.output_dir}')

In [ ]:
# 体制概率柱状图
# 构造适配器需要的数据格式
regime_viz_data = {
    'regime_probabilities': regime_result['regime_probability'],
}
path1 = viz.plot_regime_probability(regime_viz_data)
print(f'✅ 体制概率图: {path1}')

# 显示图片
from IPython.display import Image, display
display(Image(filename=path1))

In [ ]:
# 衍生品仪表盘
deriv_viz_data = {
    'term_structure': deriv_report.get('term_structure', {}),
    'basis_analysis': deriv_report.get('index_futures_basis', {}),
}
path2 = viz.plot_derivatives_dashboard(deriv_viz_data)
print(f'✅ 衍生品仪表盘: {path2}')
display(Image(filename=path2))

In [ ]:
# 风险仪表盘(使用模拟数据演示)
risk_viz_data = {
    'overall_risk_score': 42,
    'risk_level': '中低',
    'risk_factors': {
        '估值风险': 35,
        '流动性风险': 28,
        '波动率风险': 55,
        '杠杆风险': 30,
        '集中度风险': 25,
    },
    'risk_metrics': {
        'VaR_95': -2.35,
        'CVaR_95': -3.12,
        '最大回撤': -8.5,
        '波动率': 18.2,
        '夏普比率': 0.85,
    },
}
path3 = viz.plot_risk_dashboard(risk_viz_data)
print(f'✅ 风险仪表盘: {path3}')
display(Image(filename=path3))

## 2. PE 百分位历史走势图

展示沪深300/中证500/中证1000 的PE百分位历史走势

In [ ]:
# PE百分位历史走势图
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

pe_codes = [
    ('000300', '沪深300', '#3498db'),
    ('000905', '中证500', '#e74c3c'),
    ('000852', '中证1000', '#2ecc71'),
]

for ax, (code, name, color) in zip(axes, pe_codes):
    df_pe = data_svc.load_pe_data(code)
    if not df_pe.empty and 'pe_ttm' in df_pe.columns:
        pe_series = df_pe.set_index('date')['pe_ttm'].dropna()
        
        # 计算滚动百分位
        window = min(500, len(pe_series))
        rolling_pct = pe_series.rolling(window).apply(
            lambda x: (x.iloc[-1] > x).sum() / len(x) * 100
        )
        
        ax.plot(rolling_pct.index, rolling_pct.values, color=color, linewidth=1.2, label=f'{name} PE百分位')
        ax.axhline(25, color='#2ecc71', linestyle='--', alpha=0.5, label='低估线(25%)')
        ax.axhline(75, color='#e74c3c', linestyle='--', alpha=0.5, label='高估线(75%)')
        ax.fill_between(rolling_pct.index, 25, 75, alpha=0.05, color='gray')
        ax.set_ylabel('PE百分位(%)', fontsize=10)
        ax.set_title(f'{name} PE百分位走势', fontsize=12, fontweight='bold')
        ax.legend(loc='best', fontsize=8)
        ax.set_ylim(0, 100)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    else:
        ax.text(0.5, 0.5, f'{name} PE数据不可用', ha='center', va='center',
                fontsize=14, color='#999', transform=ax.transAxes)

plt.suptitle('A股核心指数 PE百分位走势', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
pe_path = str(VIZ_DIR / 'pe_percentile_history.png')
plt.savefig(pe_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ PE百分位走势图: {pe_path}')

## 3. 多指数均线趋势图

展示沪深300/中证500/中证1000/中证2000 的价格走势 + MA20/MA60

In [ ]:
# 多指数均线趋势图
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

benchmark_indices = [
    ('000300', '沪深300', 'index_sh', '#3498db'),
    ('000905', '中证500', 'index_sh', '#e74c3c'),
    ('000852', '中证1000', 'index_sh', '#2ecc71'),
    ('932000', '中证2000', 'index_sh', '#9b59b6'),
]

for ax, (code, name, mtype, color) in zip(axes, benchmark_indices):
    df = data_svc.load_index_data(code, min_days=120)
    if not df.empty and len(df) >= 60:
        df['ma20'] = df['close'].rolling(20).mean()
        df['ma60'] = df['close'].rolling(60).mean()
        
        ax.plot(df.index, df['close'], color=color, linewidth=1.0, label='收盘价', alpha=0.8)
        ax.plot(df.index, df['ma20'], color='#f39c12', linewidth=1.0, label='MA20', alpha=0.7)
        ax.plot(df.index, df['ma60'], color='#e74c3c', linewidth=1.0, label='MA60', alpha=0.7)
        
        # 标注当前状态
        latest = df['close'].iloc[-1]
        above_ma20 = '站上MA20' if latest > df['ma20'].iloc[-1] else '跌破MA20'
        above_ma60 = '站上MA60' if latest > df['ma60'].iloc[-1] else '跌破MA60'
        ax.set_title(f'{name} ({above_ma20}, {above_ma60})', fontsize=11, fontweight='bold')
        ax.legend(loc='best', fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    else:
        ax.text(0.5, 0.5, f'{name} 数据不可用', ha='center', va='center',
                fontsize=14, color='#999', transform=ax.transAxes)

plt.suptitle('A股基准指数均线趋势', fontsize=14, fontweight='bold')
plt.tight_layout()
trend_path = str(VIZ_DIR / 'benchmark_trend.png')
plt.savefig(trend_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 均线趋势图: {trend_path}')

## 4. 九宫格市场状态图

将3D市场状态映射到九宫格可视化

In [ ]:
# 九宫格市场状态可视化
fig, ax = plt.subplots(figsize=(12, 8))

grid_states = [
    ('战略进攻区', 85, 15, '#2ecc71', 0.7),
    ('积极配置区', 70, 15, '#58d68d', 0.5),
    ('均衡持有区', 55, 15, '#f39c12', 0.5),
    ('防御观望区', 40, 15, '#e67e22', 0.5),
    ('战略防御区', 25, 15, '#e74c3c', 0.7),
    ('左侧布局区', 70, 45, '#3498db', 0.4),
    ('左侧防御区', 70, 55, '#9b59b6', 0.4),
    ('防御进攻区', 40, 45, '#1abc9c', 0.4),
    ('谨慎持有区', 55, 55, '#95a5a6', 0.3),
]

for name, val_x, mom_y, color, alpha in grid_states:
    rect = plt.Rectangle((val_x - 7.5, mom_y - 5), 15, 10,
                         facecolor=color, alpha=alpha, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(val_x, mom_y, name, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

# 标记当前位置
val_s = classification_result['valuation_score']
mom_s = classification_result['momentum_score']
ax.scatter(val_s, mom_s, c='red', s=200, zorder=10, edgecolors='white', linewidths=2, marker='*')
ax.annotate(f'当前: {classification_result["market_state"]}\n'
           f'估值={val_s:.0f} 动量={mom_s:.0f}',
           xy=(val_s, mom_s), xytext=(val_s + 10, mom_s + 10),
           fontsize=10, fontweight='bold', color='red',
           arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_xlim(10, 95)
ax.set_ylim(5, 70)
ax.set_xlabel('估值得分 (低← →高)', fontsize=12)
ax.set_ylabel('动量得分 (弱← →强)', fontsize=12)
ax.set_title('V7 九宫格市场状态图', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
grid_path = str(VIZ_DIR / 'market_state_grid.png')
plt.savefig(grid_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 九宫格状态图: {grid_path}')

## 5. 波动率体制可视化

展示沪深300的历史波动率及体制切换信号

In [ ]:
# 波动率体制可视化
df_hs300 = data_svc.load_index_data('000300', min_days=500)

if not df_hs300.empty and len(df_hs300) >= 60:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                                         gridspec_kw={'height_ratios': [2, 1, 1]})
    
    # 价格 + MA
    df_hs300['ma20'] = df_hs300['close'].rolling(20).mean()
    df_hs300['ma60'] = df_hs300['close'].rolling(60).mean()
    ax1.plot(df_hs300.index, df_hs300['close'], color='#3498db', linewidth=0.8, label='沪深300')
    ax1.plot(df_hs300.index, df_hs300['ma20'], color='#f39c12', linewidth=0.8, label='MA20')
    ax1.plot(df_hs300.index, df_hs300['ma60'], color='#e74c3c', linewidth=0.8, label='MA60')
    ax1.set_ylabel('价格', fontsize=10)
    ax1.set_title('沪深300 价格走势与均线', fontsize=12, fontweight='bold')
    ax1.legend(loc='best', fontsize=8)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # 滚动波动率
    returns = df_hs300['close'].pct_change()
    vol_20 = returns.rolling(20).std() * np.sqrt(252) * 100
    vol_60 = returns.rolling(60).std() * np.sqrt(252) * 100
    ax2.plot(vol_20.index, vol_20.values, color='#3498db', linewidth=0.8, label='20d波动率')
    ax2.plot(vol_60.index, vol_60.values, color='#e74c3c', linewidth=0.8, label='60d波动率')
    ax2.axhline(15, color='#2ecc71', linestyle='--', alpha=0.5, label='牛市阈值(15%)')
    ax2.axhline(25, color='#e74c3c', linestyle='--', alpha=0.5, label='震荡阈值(25%)')
    ax2.axhline(30, color='#9b59b6', linestyle='--', alpha=0.5, label='熊市阈值(30%)')
    ax2.set_ylabel('年化波动率(%)', fontsize=10)
    ax2.set_title('滚动波动率', fontsize=12, fontweight='bold')
    ax2.legend(loc='best', fontsize=8)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    # 动量
    mom_20 = returns.rolling(20).sum() * 100
    mom_60 = returns.rolling(60).sum() * 100
    ax3.plot(mom_20.index, mom_20.values, color='#3498db', linewidth=0.8, label='20d动量')
    ax3.plot(mom_60.index, mom_60.values, color='#e74c3c', linewidth=0.8, label='60d动量')
    ax3.axhline(0, color='gray', linewidth=0.5)
    ax3.axhline(5, color='#2ecc71', linestyle='--', alpha=0.5, label='牛市阈值(5%)')
    ax3.axhline(-5, color='#e74c3c', linestyle='--', alpha=0.5, label='熊市阈值(-5%)')
    ax3.fill_between(mom_20.index, -2, 2, alpha=0.1, color='gray', label='震荡区间')
    ax3.set_ylabel('累计收益率(%)', fontsize=10)
    ax3.set_title('动量(累计收益率)', fontsize=12, fontweight='bold')
    ax3.legend(loc='best', fontsize=8)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    
    plt.suptitle('市场体制分析: 波动率 + 动量', fontsize=14, fontweight='bold')
    plt.tight_layout()
    vol_path = str(VIZ_DIR / 'volatility_regime.png')
    plt.savefig(vol_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 波动率体制图: {vol_path}')
else:
    print('⚠️ 沪深300数据不足')

## 6. 产业景气度雷达图

九大战略方向的景气度雷达图

In [ ]:
# 产业景气度雷达图
industry_sentiment = deriv_report.get('industry_sentiment', {})

if industry_sentiment:
    labels = list(industry_sentiment.keys())
    values = list(industry_sentiment.values())
    n = len(labels)
    
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    values_closed = values + [values[0]]
    angles_closed = angles + [angles[0]]
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
    
    ax.plot(angles_closed, values_closed, 'o-', color='#3498db', linewidth=2, markersize=8)
    ax.fill(angles_closed, values_closed, alpha=0.2, color='#3498db')
    
    # 50分中性线
    ax.plot(angles_closed, [50] * len(angles_closed), '--', color='gray', alpha=0.5, linewidth=1)
    
    ax.set_xticks(angles)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylim(0, 100)
    ax.set_title('九大战略方向产业景气度', fontsize=14, fontweight='bold', pad=20)
    
    # 标注数值
    for angle, value, label in zip(angles, values, labels):
        color = '#2ecc71' if value > 65 else ('#f39c12' if value > 50 else '#e74c3c')
        ax.annotate(f'{value:.0f}', xy=(angle, value), xytext=(0, 10),
                    textcoords='offset points', ha='center', fontsize=9, fontweight='bold', color=color)
    
    plt.tight_layout()
    radar_path = str(VIZ_DIR / 'industry_sentiment_radar.png')
    plt.savefig(radar_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 产业景气度雷达图: {radar_path}')
else:
    print('⚠️ 产业景气度数据不可用')

## 7. V7 综合市场状态仪表盘

将核心指标合并为一张综合仪表盘

In [ ]:
# V7 综合市场状态仪表盘
fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(3, 4, hspace=0.35, wspace=0.35)

# ── [0,0:2] 市场状态摘要 ──
ax_summary = fig.add_subplot(gs[0, 0:2])
ax_summary.axis('off')
state = classification_result['market_state']
composite = classification_result['composite_score']
regime_cn = classification_result['regime_name']
confidence = classification_result['confidence']

# 状态颜色
state_colors = {
    '战略进攻区': '#2ecc71', '积极配置区': '#58d68d',
    '均衡持有区': '#f39c12', '防御观望区': '#e67e22',
    '战略防御区': '#e74c3c', '左侧布局区': '#3498db',
    '左侧防御区': '#9b59b6', '防御进攻区': '#1abc9c',
    '谨慎持有区': '#95a5a6',
}
state_color = state_colors.get(state, '#95a5a6')

ax_summary.add_patch(plt.Rectangle((0.05, 0.1), 0.9, 0.85, facecolor=state_color, alpha=0.2))
ax_summary.text(0.5, 0.7, state, ha='center', va='center', fontsize=28, fontweight='bold', color=state_color)
ax_summary.text(0.5, 0.4, f'综合得分: {composite:.1f} | 体制: {regime_cn} | 置信度: {confidence:.0%}',
               ha='center', va='center', fontsize=12, color='#333')
ax_summary.text(0.5, 0.2, f'估值={classification_result["valuation_score"]:.0f} '
               f'动量={classification_result["momentum_score"]:.0f} '
               f'体制={classification_result["regime_score"]:.0f}',
               ha='center', va='center', fontsize=11, color='#666')
ax_summary.set_title('V7 市场状态', fontsize=14, fontweight='bold')

# ── [0,2:4] 体制概率 ──
ax_regime = fig.add_subplot(gs[0, 2:4])
regime_names_cn = {'bull': '牛市', 'bear': '熊市', 'volatile': '震荡', 'recovery': '复苏'}
probs = regime_result['regime_probability']
labels_r = [regime_names_cn.get(k, k) for k in probs.keys()]
values_r = list(probs.values())
colors_r = ['#3498db', '#e74c3c', '#f39c12', '#2ecc71']
bars = ax_regime.bar(labels_r, values_r, color=colors_r[:len(labels_r)], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, values_r):
    ax_regime.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                   f'{val:.1%}', ha='center', va='bottom', fontsize=10)
ax_regime.set_title('体制概率', fontsize=14, fontweight='bold')
ax_regime.set_ylim(0, max(values_r) * 1.3 if max(values_r) > 0 else 1)
ax_regime.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax_regime.spines['top'].set_visible(False)
ax_regime.spines['right'].set_visible(False)

# ── [1,0:2] 基差分析 ──
ax_basis = fig.add_subplot(gs[1, 0:2])
basis_data = deriv_report.get('index_futures_basis', {})
if basis_data:
    names_b = list(basis_data.keys())
    basis_vals = [basis_data[n].get('basis_pct', 0) for n in names_b]
    colors_b = ['#e74c3c' if v < 0 else '#2ecc71' for v in basis_vals]
    ax_basis.bar(names_b, basis_vals, color=colors_b, edgecolor='white')
    ax_basis.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax_basis.set_title('股指期货基差(%)', fontsize=13, fontweight='bold')
ax_basis.spines['top'].set_visible(False)
ax_basis.spines['right'].set_visible(False)

# ── [1,2:4] 产业景气度TOP5 ──
ax_ind = fig.add_subplot(gs[1, 2:4])
if industry_sentiment:
    sorted_sent = sorted(industry_sentiment.items(), key=lambda x: x[1], reverse=True)
    top5_names = [s[0] for s in sorted_sent[:5]]
    top5_vals = [s[1] for s in sorted_sent[:5]]
    colors_i = ['#2ecc71' if v > 65 else ('#f39c12' if v > 50 else '#e74c3c') for v in top5_vals]
    ax_ind.barh(top5_names[::-1], top5_vals[::-1], color=colors_i[::-1], edgecolor='white')
    for i, (n, v) in enumerate(zip(top5_names[::-1], top5_vals[::-1])):
        ax_ind.text(v + 1, i, f'{v:.0f}', va='center', fontsize=9)
ax_ind.set_title('产业景气度 TOP5', fontsize=13, fontweight='bold')
ax_ind.spines['top'].set_visible(False)
ax_ind.spines['right'].set_visible(False)

# ── [2,0:2] 期限结构 ──
ax_ts = fig.add_subplot(gs[2, 0:2])
term_data = deriv_report.get('term_structure', {})
if term_data:
    ts_keys = list(term_data.keys())
    spreads = [term_data[k]['spread'] for k in ts_keys]
    colors_ts = ['#2ecc71' if s > 0 else '#e74c3c' for s in spreads]
    ax_ts.bar(ts_keys, spreads, color=colors_ts, edgecolor='white')
    ax_ts.axhline(0, color='#555', linewidth=0.8, linestyle='--')
ax_ts.set_title('期限结构价差(%)', fontsize=13, fontweight='bold')
ax_ts.spines['top'].set_visible(False)
ax_ts.spines['right'].set_visible(False)

# ── [2,2:4] 商品信号 ──
ax_comm = fig.add_subplot(gs[2, 2:4])
comm_signals = deriv_report.get('commodity_signals', {})
if comm_signals:
    comm_names = [v['name'] for v in comm_signals.values()]
    comm_chg = [v['price_chg_20d'] for v in comm_signals.values()]
    colors_c = ['#e74c3c' if v < 0 else '#2ecc71' for v in comm_chg]
    ax_comm.barh(comm_names[::-1], comm_chg[::-1], color=colors_c[::-1], edgecolor='white')
    ax_comm.axvline(0, color='#555', linewidth=0.8, linestyle='--')
ax_comm.set_title('商品期货20d涨跌(%)', fontsize=13, fontweight='bold')
ax_comm.spines['top'].set_visible(False)
ax_comm.spines['right'].set_visible(False)

fig.suptitle(f'AiStock V7 综合市场状态仪表盘 — {datetime.now().strftime("%Y-%m-%d")}',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
dashboard_path = str(VIZ_DIR / 'v7_dashboard.png')
plt.savefig(dashboard_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 综合仪表盘: {dashboard_path}')

## 8. 可视化输出汇总

In [ ]:
# 列出所有生成的可视化文件
import glob

viz_files = sorted(glob.glob(str(VIZ_DIR / '*.png')))
print(f'可视化输出目录: {VIZ_DIR}')
print(f'生成文件数: {len(viz_files)}')
print(f'\n文件列表:')
total_size = 0
for f in viz_files:
    size_kb = os.path.getsize(f) / 1024
    total_size += size_kb
    print(f'  {os.path.basename(f):<40} {size_kb:>8.1f} KB')
print(f'\n总大小: {total_size:.1f} KB')

In [ ]:
# 清理资源
tdx.close()
db.close()
ak.close()
print('✅ 所有资源已释放')
print(f'\n可视化测试完成! 共生成 {len(glob.glob(str(VIZ_DIR / "*.png")))} 张图表')